In [1]:
import os
from pathlib import Path

os.getcwd()
mb_dir = Path(os.getcwd()).parent.parent
os.chdir(mb_dir)
data_dir = str(mb_dir.parent / "data")
root_dir = str(mb_dir.parent)

In [2]:
import xarray as xr
import scipy.io as sio
import numpy as np

# file_30day = f"{root_dir}/fig_data/deterministic_scores_30_day_2019_2024.mat"
# data_30 = sio.loadmat(file_30day)

# print(data_30.keys())
# print(data_30["mae_cmz_fixed_clim"])
# print(data_30["std_er_fixed_clim"])
# print(data_30["std_er_30"])

In [19]:
import numpy as np

from examples.paper_figures.data_utils import (
    get_model_dfs, get_plot_metrics, save_data,
    load_wyi, get_climatological_dfs, load_wyi,
    YEAR_RANGES, EXTENDED_YEARS, YEAR_RANGES_COM
)

# Figure 4 year ranges
YEAR_RANGES_COM = {
    "AIFS": np.arange(2004, 2022),
    "IFS": np.arange(2004, 2022),
    "FuXi": np.arange(2004, 2022),
    "Graphcast": np.arange(2004, 2022),
    "GenCast": np.arange(2019, 2022),
    "FuXi-S2S": np.arange(2004, 2022),
    "NGCM": np.arange(2004, 2022),
}

config = {
    "years": np.arange(2019, 2025),
    "extended_years": np.concatenate((np.arange(1965, 1979), np.arange(2019, 2025))),  # Extended period for analysis
    "common_years": np.arange(2004, 2022),
    "imd_folder": f"{data_dir}/imd_rainfall_data/4p0",  # Ground truth rainfall data (4x4 degrees)
    "thres_file": f"{data_dir}/imd_onset_threshold/mwset4x4.nc4",  # Threshold for the onset of the monsoon (4x4 degrees)
    "shpfile_path": f"{data_dir}/ind_map_shpfile/india_shapefile.shp",  # Shapefile of India
    "output_dir": f"{root_dir}/output",  # Directory to save data files
}

config2 = {
    "imd_folder": f"{data_dir}/imd_rainfall_data/4p0",  # Ground truth rainfall data (4x4 degrees)
    "thres_file": f"{data_dir}/imd_onset_threshold/mwset4x4.nc4",  # Threshold for the onset of the monsoon (4x4 degrees)
    "shpfile_path": f"{data_dir}/ind_map_shpfile/india_shapefile.shp",  # Shapefile of India
    "output_dir": f"{root_dir}/output",  # Directory to save data files
    "years": np.concatenate((np.arange(1965, 1979), np.arange(2019, 2025)))  # Extended period for analysis
}


config3 = {
    "imd_folder": f"{data_dir}/imd_rainfall_data/4p0",  # Ground truth rainfall data (4x4 degrees)
    "thres_file": f"{data_dir}/imd_onset_threshold/mwset4x4.nc4",  # Threshold for the onset of the monsoon (4x4 degrees)
    "shpfile_path": f"{data_dir}/ind_map_shpfile/india_shapefile.shp",  # Shapefile of India
    "output_dir": f"{root_dir}/output",  # Directory to save data files
    "years": np.arange(2004, 2022)  # Extended period for analysis
}

model_paths = {
    "IFS": f"{data_dir}/rainfall_4p0/IFS_S2S",
    "AIFS":  f"{data_dir}/rainfall_4p0/AIFS",
    "FuXi": f"{data_dir}/rainfall_4p0/FuXi",
    "Graphcast": f"{data_dir}/rainfall_4p0/GraphCast",
    "GenCast": f"{data_dir}/rainfall_4p0/GenCast",
    "FuXi-S2S": f"{data_dir}/rainfall_4p0/FuXi_S2S",
    "NGCM": f"{data_dir}/rainfall_4p0/NeuralGCM"
}

prob_paths = {
    "NGCM": f"{data_dir}/rainfall_4p0/NeuralGCM"
}

prob_paths.keys()

dict_keys(['NGCM'])

### Testing Issue 1: Wrong handling of probabilistic models

In [20]:
# Compute 30-day forecast data for the extended period
from monsoonbench.metrics import ClimatologyOnsetMetrics

c_metrics = ClimatologyOnsetMetrics()

print("Loading Climatoligcal Data")
clim_data = get_climatological_dfs(config=config)
clim_df_15, clim_onset_15 = clim_data["15_day"]
clim_df_15_ex, clim_onset_15_ex = clim_data["15_day_ex"]
clim_df_30, clim_onset_30 = clim_data["30_day"]
clim_df_30_ex, clim_onset_30_ex = clim_data["30_day_ex"]

Loading Climatoligcal Data
Computing climatological onset reference...
Computing climatological onset from 124 years: 1901-2024
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1901.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1901-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1902.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1902-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1903.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1903-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1904.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1904-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\

In [21]:
from monsoonbench.metrics import ProbabilisticOnsetMetrics, DeterministicOnsetMetrics
import pandas as pd

DETERMINISTIC_MODELS = ['IFS', 'AIFS', 'FuXi', 'Graphcast']
PROBABILISTIC_MODELS = ['GenCast', 'FuXi-S2S', 'NGCM']


model_df_prob = get_model_dfs(
    model_paths=prob_paths,
    year_ranges=YEAR_RANGES,
    config=config,
    days=15
)



Processing year 2019
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\2019.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (2019-06-02) as start date for onset detection
Processing 26 init times x 8 lats x 9 lons...
Using MOK (6/2 filter) for onset detection
Only processing forecasts initialized before observed onset dates
Requiring ≥50% of 51 members to have onset for ensemble onset
Processing init time 1/26: 2019-05-02
Processing init time 6/26: 2019-05-20
Processing init time 11/26: 2019-06-06
Processing init time 16/26: 2019-06-24
Processing init time 21/26: 2019-07-11
Processing init time 26/26: 2019-07-29

Processing Summary:
Total potential initializations: 1872
Skipped (no observed onset): 962
Skipped (initialized after observed onset): 394
Valid initializations processed: 516
Ensemble onsets found (≥50% members): 142
Ensemble onset rate: 0.275
Note: Only onsets on or after 6/2 were counted due to MOK flag
Computing o

In [22]:
from monsoonbench.metrics import (
    ClimatologyOnsetMetrics,
    DeterministicOnsetMetrics,
    ProbabilisticOnsetMetrics,
)
from monsoonbench.visualization import (
    create_model_comparison_table,
)

prob_dfs_15, prob_onsets_15 = model_df_prob
md_15 = get_plot_metrics(
    prob_dfs_15, prob_onsets_15, clim_df_15, clim_onset_15, config["extended_years"], 15
)

Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [23]:
print(md_15.keys())

print(md_15["mae_cmz_mean_15"])


dict_keys(['false_alarm_15', 'far_cmz_mean_15', 'lat', 'lon', 'mae_avg_15', 'mae_cmz_fixed_clim', 'mae_cmz_mean_15', 'mae_yr_15', 'miss_rate_15', 'mr_cmz_mean_15', 'std_er_15', 'std_er_fixed_clim'])
[5.68293651 3.39499108]


In [13]:
plot_metrics = {}
c_metrics = ClimatologyOnsetMetrics()

plot_probabilistic_metrics = c_metrics.create_spatial_far_mr_mae(
    prob_dfs_15["IFS"], prob_onsets_15["IFS"]
)
plot_metrics["IFS"] = plot_probabilistic_metrics

create_model_comparison_table(plot_metrics)

Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


c:\Users\cflor\CSAssignments\Clinic3\monsoon-bench\monsoonbench\visualization\spatial.py:57: RuntimeWarning: Mean of empty slice
  return float(np.nanmean(values_inside))
c:\Users\cflor\CSAssignments\Clinic3\monsoon-bench\monsoonbench\visualization\spatial.py:105: RuntimeWarning: Mean of empty slice
  overall_avg = np.nanmean(year_mae_map.to_numpy())


,cmz_mae_mean_days,cmz_mae_se_days,cmz_far_pct,cmz_mr_pct,overall_mae_mean_days,overall_far_pct,overall_mr_pct
model,,,,,,,
IFS,NaN,NaN,0.0,100.0,NaN,0.0,100.0


Added climatology 15, 30 day for common period to data_utils

In [44]:
import numpy as np
import pandas as pd
import xarray as xr
from scipy.io import savemat

from monsoonbench.metrics import (
    ClimatologyOnsetMetrics,
    DeterministicOnsetMetrics,
    ProbabilisticOnsetMetrics,
)
from monsoonbench.visualization import (
    create_model_comparison_table,
)

def get_plot_metrics(
    model_dfs: dict[str, pd.DataFrame],
    model_onsets: dict[str, xr.DataArray],
    metrics_df_clim: pd.DataFrame,
    onset_da_clim: xr.DataArray,
    year_range: list[int],
    day: int = 15,
) -> dict[str, np.ndarray]:
    """Get Figure 1 & 4 plot metrics for a given set of model

    dataframes, onset data arrays, climatological data, and configuration.

    Args:
        model_dfs: Dictionary of model names and their individual metric dataframes.
        model_onsets: Dictionary of model names and their onset data arrays.
        metrics_df_clim: Climatological metrics dataframe.
        onset_da_clim: Climatological onset data array.
        config: Dictionary of configuration parameters.
        day: Number of days to forecast (15 or 30).

    Returns:
        Dictionary of plot metrics.
    """
    plot_metrics = {}
    c_metrics = ClimatologyOnsetMetrics()

    for model_name in model_dfs.keys():
        probabilistic_df = model_dfs[model_name]
        onset_da_dict = model_onsets[model_name]
        plot_probabilistic_metrics = c_metrics.create_spatial_far_mr_mae(
            probabilistic_df, onset_da_dict
        )
        plot_metrics[model_name] = plot_probabilistic_metrics

    clim_plot_data = c_metrics.create_spatial_far_mr_mae(
        metrics_df_clim, dict.fromkeys(year_range, onset_da_clim)
    )

    mean_mae = plot_metrics["AIFS"]["mean_mae"]

    # Get coordinates
    lats = mean_mae.lat.to_numpy()
    lons = mean_mae.lon.to_numpy()

    false_alarm_15 = [clim_plot_data["false_alarm_rate"]]
    miss_rate_15 = [clim_plot_data["miss_rate"]]
    mae_yr_15 = [clim_plot_data["mean_mae"]]

    for model_name in plot_metrics.keys():
        for stat in plot_metrics[model_name].keys():
            if "miss_rate" in stat:
                miss_rate_15.append(plot_metrics[model_name][stat])
            if "mean_mae" in stat:
                mae_yr_15.append(plot_metrics[model_name][stat])
            if "false" in stat:
                false_alarm_15.append(plot_metrics[model_name][stat])

    mae_yr_15 = xr.concat(mae_yr_15, dim="stack").transpose()
    miss_rate_15 = xr.concat(miss_rate_15, dim="stack").transpose()
    false_alarm_15 = xr.concat(false_alarm_15, dim="stack").transpose()
    plot_metrics["clim"] = clim_plot_data

    cmz_metrics = create_model_comparison_table(plot_metrics)

    cmz_metrics = pd.concat([cmz_metrics.tail(1), cmz_metrics.iloc[:-1]])

    # Format output dictionary
    mat_dict = {
        f"false_alarm_{str(day)}": false_alarm_15.values,
        f"far_cmz_mean_{str(day)}": np.array(cmz_metrics["cmz_far_pct"].values),
        "lat": lats,
        "lon": lons,
        f"mae_avg_{str(day)}": mae_yr_15.values,
        "mae_cmz_fixed_clim": np.array(
            [[7.18333333]]
        ),  # Fixed climatological MAE for 15-day and 30-day forecasts
        f"mae_cmz_mean_{str(day)}": np.array(cmz_metrics["cmz_mae_mean_days"].values),
        f"mae_yr_{str(day)}": np.array(cmz_metrics["overall_mae_mean_days"].values),
        f"miss_rate_{str(day)}": miss_rate_15.values,
        f"mr_cmz_mean_{str(day)}": np.array(cmz_metrics["cmz_mr_pct"].values),
        f"std_er_{str(day)}": np.array(cmz_metrics["cmz_mae_se_days"].values),
        "std_er_fixed_clim": np.array(
            [[0.9686474]]
        ),  # Fixed climatological standard error for 15-day and 30-day forecasts
    }

    return mat_dict



Loading Climatoligcal Data
Computing climatological onset reference...
Computing climatological onset from 124 years: 1901-2024
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1901.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1901-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1902.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1902-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1903.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1903-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1904.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1904-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\

In [7]:
model_dfs_15, model_onsets_15 = get_model_dfs(
    model_paths, year_ranges=YEAR_RANGES, config=config, days=15
)



Processing year 2019
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\2019.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (2019-06-02) as start date for onset detection
Processing 26 init times x 8 lats x 9 lons...
Using MOK (6/2 filter) for onset detection
Only processing forecasts initialized before observed onset dates
Requiring ≥50% of 11 members to have onset for ensemble onset
Processing init time 1/26: 2019-05-02
Processing init time 6/26: 2019-05-20
Processing init time 11/26: 2019-06-06
Processing init time 16/26: 2019-06-24
Processing init time 21/26: 2019-07-11
Processing init time 26/26: 2019-07-29

Processing Summary:
Total potential initializations: 1872
Skipped (no observed onset): 962
Skipped (initialized after observed onset): 394
Valid initializations processed: 516
Ensemble onsets found (≥50% members): 141
Ensemble onset rate: 0.273
Note: Only onsets on or after 6/2 were counted due to MOK flag
Computing o

In [8]:
model_dfs_30, model_onsets_30 = get_model_dfs(
    model_paths, year_ranges=YEAR_RANGES, config=config, days=30
)



Processing year 2019
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\2019.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (2019-06-02) as start date for onset detection
Processing 26 init times x 8 lats x 9 lons...
Using MOK (6/2 filter) for onset detection
Only processing forecasts initialized before observed onset dates
Requiring ≥50% of 11 members to have onset for ensemble onset
Processing init time 1/26: 2019-05-02
Processing init time 6/26: 2019-05-20
Processing init time 11/26: 2019-06-06
Processing init time 16/26: 2019-06-24
Processing init time 21/26: 2019-07-11
Processing init time 26/26: 2019-07-29

Processing Summary:
Total potential initializations: 1872
Skipped (no observed onset): 962
Skipped (initialized after observed onset): 394
Valid initializations processed: 516
Ensemble onsets found (≥50% members): 272
Ensemble onset rate: 0.527
Note: Only onsets on or after 6/2 were counted due to MOK flag
Computing o

In [19]:
model_dfs_15_ex, model_onsets_15_ex = get_model_dfs(
    model_paths, year_ranges=EXTENDED_YEARS, config=config, days=15
)



Processing year 2013
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\2013.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (2013-06-02) as start date for onset detection
Processing 26 init times x 8 lats x 9 lons...
Using MOK (6/2 filter) for onset detection
Only processing forecasts initialized before observed onset dates
Requiring ≥50% of 11 members to have onset for ensemble onset
Processing init time 1/26: 2013-05-02
Processing init time 6/26: 2013-05-20
Processing init time 11/26: 2013-06-06
Processing init time 16/26: 2013-06-24
Processing init time 21/26: 2013-07-11
Processing init time 26/26: 2013-07-29

Processing Summary:
Total potential initializations: 1872
Skipped (no observed onset): 962
Skipped (initialized after observed onset): 465
Valid initializations processed: 445
Ensemble onsets found (≥50% members): 183
Ensemble onset rate: 0.411
Note: Only onsets on or after 6/2 were counted due to MOK flag
Computing o

In [23]:
model_dfs_30_ex, model_onsets_30_ex = get_model_dfs(
    model_paths, year_ranges=EXTENDED_YEARS, config=config, days=30
)




Processing year 2013
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\2013.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (2013-06-02) as start date for onset detection
Processing 26 init times x 8 lats x 9 lons...
Using MOK (6/2 filter) for onset detection
Only processing forecasts initialized before observed onset dates
Requiring ≥50% of 11 members to have onset for ensemble onset
Processing init time 1/26: 2013-05-02
Processing init time 6/26: 2013-05-20
Processing init time 11/26: 2013-06-06
Processing init time 16/26: 2013-06-24
Processing init time 21/26: 2013-07-11
Processing init time 26/26: 2013-07-29

Processing Summary:
Total potential initializations: 1872
Skipped (no observed onset): 962
Skipped (initialized after observed onset): 465
Valid initializations processed: 445
Ensemble onsets found (≥50% members): 302
Ensemble onset rate: 0.679
Note: Only onsets on or after 6/2 were counted due to MOK flag
Computing o

In [21]:
model_dfs_15_cm, model_onsets_15_cm = get_model_dfs(
    model_paths, year_ranges=YEAR_RANGES_COM, config=config, days=15
)



Processing year 2004
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\2004.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (2004-06-02) as start date for onset detection
Processing 26 init times x 8 lats x 9 lons...
Using MOK (6/2 filter) for onset detection
Only processing forecasts initialized before observed onset dates
Requiring ≥50% of 11 members to have onset for ensemble onset
Processing init time 1/26: 2004-05-02
Processing init time 6/26: 2004-05-20
Processing init time 11/26: 2004-06-06
Processing init time 16/26: 2004-06-24
Processing init time 21/26: 2004-07-11
Processing init time 26/26: 2004-07-29

Processing Summary:
Total potential initializations: 1872
Skipped (no observed onset): 936
Skipped (initialized after observed onset): 470
Valid initializations processed: 466
Ensemble onsets found (≥50% members): 160
Ensemble onset rate: 0.343
Note: Only onsets on or after 6/2 were counted due to MOK flag
Computing o

In [ ]:

model_dfs_30_cm, model_onsets_30_cm = get_model_dfs(
    model_paths, year_ranges=YEAR_RANGES_COM, config=config, days=30
)


Processing year 2004
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\2004.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (2004-06-02) as start date for onset detection
Processing 26 init times x 8 lats x 9 lons...
Using MOK (6/2 filter) for onset detection
Only processing forecasts initialized before observed onset dates
Requiring ≥50% of 11 members to have onset for ensemble onset
Processing init time 1/26: 2004-05-02
Processing init time 6/26: 2004-05-20
Processing init time 11/26: 2004-06-06
Processing init time 16/26: 2004-06-24
Processing init time 21/26: 2004-07-11
Processing init time 26/26: 2004-07-29

Processing Summary:
Total potential initializations: 1872
Skipped (no observed onset): 936
Skipped (initialized after observed onset): 470
Valid initializations processed: 466
Ensemble onsets found (≥50% members): 286
Ensemble onset rate: 0.614
Note: Only onsets on or after 6/2 were counted due to MOK flag
Computing o

In [98]:
md_15 = get_plot_metrics(
    model_dfs_15, model_onsets_15, clim_df_15, clim_onset_15, config["years"], 15
)
save_data(md_15, config["output_dir"], "deterministic_scores_15_day_2019_2024")

md_30 = get_plot_metrics(
    model_dfs_30, model_onsets_30, clim_df_30, clim_onset_30, config["years"], 30
)
save_data(md_30, config["output_dir"], "deterministic_scores_30_day_2019_2024")



Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
C

In [82]:
type(model_dfs_15)

dict

In [47]:
md_15_ex = get_plot_metrics(
    model_dfs_15_ex, model_onsets_15_ex, clim_df_15_ex, clim_onset_15_ex, config2["years"], 15
)
save_data(
    md_15_ex,
    config["output_dir"],
    "deterministic_scores_15_day_1965_1978_2019_2024_with_gencast",
)

md_30_ex = get_plot_metrics(
    model_dfs_30_ex, model_onsets_30_ex, clim_df_30_ex, clim_onset_30_ex, config2["years"], 30
)
save_data(
    md_30_ex,
    config["output_dir"],
    "deterministic_scores_30_day_1965_1978_2019_2024_with_gencast",
)

Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [np.int64(1965), np.int64(1966), np.int64(1967), np.int64(1968), np.int64(1969), np.int64(1970), np.int64(1971), np.int64(1972), np.int64(1973), np.int64(1974), np.int64(1975), np.int64(1976), np.int64(1977), np.int64(1978), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [np.int64(1965), np.int64(1966), np.int64(1967), np.int64(1968), np.int64(1969), np.int64(1970), np.int64(1971), np.int64(1972), np.int64(1973), np.int64(1974), np.int64(1975), np.int6

In [40]:

md_15_cm = get_plot_metrics(
    model_dfs_15_cm, model_onsets_15_cm, clim_df_15_cm, clim_onset_15_cm, config3, 15
)
save_data(md_15_cm, config["output_dir"], "deterministic_scores_15_day_2004_2021")

md_30_cm = get_plot_metrics(
    model_dfs_30_cm, model_onsets_30_cm, clim_df_30_cm, clim_onset_30_cm, config3, 30
)
save_data(md_30_cm, config["output_dir"], "deterministic_scores_30_day_2004_2021")


Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int6

In [85]:
clim_plot_data_30 = c_metrics.create_spatial_far_mr_mae(
    clim_df_30, dict.fromkeys(config["years"], clim_onset_30)
)

clim_plot_data_15 = c_metrics.create_spatial_far_mr_mae(
    clim_df_15, dict.fromkeys(config["years"], clim_onset_15)
)

clim_df_30
clim_df_15

Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


{np.int64(2019):      lat   lon  total_forecasts  true_positive  true_negative  false_positive  \
 0    8.0  76.0                9              3              5               0   
 1    8.0  80.0               26              1             25               0   
 2   12.0  76.0               11              4              6               0   
 3   12.0  80.0               20              0             12               4   
 4   16.0  72.0               11              4              7               0   
 5   16.0  76.0               11              5              6               0   
 6   16.0  80.0               15              0              8               5   
 7   16.0  84.0               10              0              5               1   
 8   20.0  68.0               12              0              8               1   
 9   20.0  72.0               12              0              7               3   
 10  20.0  76.0               14              0              8               5   


In [42]:
load_wyi(config["output_dir"])

Saved to: c:\Users\cflor\CSAssignments\Clinic3/output/wyi_onset_deterministic_metrics_15day_2019_2024.mat
Saved to: c:\Users\cflor\CSAssignments\Clinic3/output/wyi_onset_deterministic_metrics_30day_2019_2024.mat
Saved to: c:\Users\cflor\CSAssignments\Clinic3/output/wyi_onset_deterministic_metrics_15day_1965_1978_2019_2024_with_gencast.mat
Saved to: c:\Users\cflor\CSAssignments\Clinic3/output/wyi_onset_deterministic_metrics_30day_1965_1978_2019_2024_with_gencast.mat
Saved to: c:\Users\cflor\CSAssignments\Clinic3/output/wyi_onset_deterministic_metrics_15day_2004_2021.mat
Saved to: c:\Users\cflor\CSAssignments\Clinic3/output/wyi_onset_deterministic_metrics_30day_2004_2021.mat


### Ground Truth (Fig 1)

In [5]:
fig1_data = f"{root_dir}/fig_data/5day_forecastwindow_cmz_2019_2024.mat"
data = sio.loadmat(fig1_data)
print(data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'far_cmz', 'mae_cmz', 'model_str', 'mr_cmz', 'std_er'])


### Ground truth stats (fig 3)

In [6]:
weekly_file = f"{root_dir}/fig_data/5day_forecastwindow_cmz_2019_2024.mat"
data = sio.loadmat(weekly_file)
mae_cmz = data['mae_cmz']  # Shape should be (6, 8) for 6 time periods, 8 models
far_cmz = data['far_cmz']  # Shape should be (6, 8)
mr_cmz = data['mr_cmz']    # Shape should be (4, 8) for 4 weeks
std_er = data['std_er']    # Standard errors for MAE

print(data["model_str"])
mae_cmz
for r in mae_cmz:
    print(r[3])

data["model_str"]

[[array(['clim'], dtype='<U4')]
 [array(['ifs'], dtype='<U3')]
 [array(['aifs'], dtype='<U4')]
 [array(['fuxi'], dtype='<U4')]
 [array(['graphcast'], dtype='<U9')]
 [array(['gencast'], dtype='<U7')]
 [array(['fuxis2s'], dtype='<U7')]
 [array(['ngcm51'], dtype='<U6')]]
1.5187169312169315
2.42744708994709
4.357936507936508
5.64917328042328
7.873848104056438
9.218915343915345


array([[array(['clim'], dtype='<U4')],
       [array(['ifs'], dtype='<U3')],
       [array(['aifs'], dtype='<U4')],
       [array(['fuxi'], dtype='<U4')],
       [array(['graphcast'], dtype='<U9')],
       [array(['gencast'], dtype='<U7')],
       [array(['fuxis2s'], dtype='<U7')],
       [array(['ngcm51'], dtype='<U6')]], dtype=object)

### Fig 3 data reproduction


In [71]:
c_metrics = ClimatologyOnsetMetrics()

def compute_climatology_baseline_multiple_years(
    years,
    imd_folder,
    thres_file,
    tolerance_days=3,
    verification_window=1,
    forecast_days=15,
    max_forecast_day=15,
    mok=True,
    onset_window=5,
    mok_month=6,
    mok_day=2,
):
    """Compute climatology baseline metrics for multiple years.

    Returns:
    metrics_df_dict: dict, {year: metrics_df}
    climatological_onset_doy: xarray DataArray with climatological onset day of year
    """
    print("Computing climatological onset reference...")

    # Compute climatological onset once (using all available years)
    climatological_onset_doy = c_metrics.compute_climatological_onset(
        imd_folder, thres_file, mok=mok
    )

    # Load threshold data
    thresh_ds = xr.open_dataset(thres_file)
    thres_da = thresh_ds["MWmean"]

    metrics_df_dict = {}

    for year in years:
        print(f"\n{'=' * 50}")
        print(f"Evaluating climatology baseline for year {year}")
        print(f"{'=' * 50}")

        # Get initialization dates for this year (same as model would use)
        init_dates = c_metrics.get_initialization_dates(year)

        # Load observed data for this year
        imd = c_metrics.load_imd_rainfall(year, imd_folder)
        observed_onset_da = c_metrics.detect_observed_onset(
            imd, thres_da, year, mok=mok
        )

        # Generate climatology forecasts for all initialization dates
        # Now passing observed_onset_da to filter initializations
        climatology_forecast_df = (
            c_metrics.compute_climatology_as_forecast(
                climatological_onset_doy,
                year,
                init_dates,
                observed_onset_da,
                max_forecast_day=max_forecast_day,
                mok=mok,
                mok_month=mok_month,
                mok_day=mok_day,
            )
        )
        metrics_df_dict[year] = climatology_forecast_df
    return metrics_df_dict, climatological_onset_doy

test_clim_forecast, clim_onset_day = compute_climatology_baseline_multiple_years(
    years=config["years"],
    imd_folder=config["imd_folder"],
    thres_file=config["thres_file"],
    tolerance_days=3,
    verification_window=1,
    forecast_days=15,
    max_forecast_day=15,
    mok=True,
    onset_window=5,
    mok_month=6,
    mok_day=2,
)


Computing climatological onset reference...
Computing climatological onset from 124 years: 1901-2024
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1901.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1901-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1902.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1902-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1903.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1903-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1904.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1904-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3

In [80]:
test_clim_forecast[2019]
clim_onset_day

<xarray.DataArray 'dayofyear' (lat: 8, lon: 9)> Size: 576B
array([[ nan,  nan, 157., 223.,  nan,  nan,  nan,  nan,  nan],
       [ nan,  nan, 157., 174.,  nan,  nan,  nan,  nan,  nan],
       [ nan, 159., 158., 165., 166.,  nan,  nan,  nan,  nan],
       [176., 167., 165., 165., 165., 164.,  nan,  nan,  nan],
       [188., 176., 173., 173., 168., 163., 157., 159.,  nan],
       [197., 182., 178., 173., 168., 159., 158., 160.,  nan],
       [ nan, 185., 175., 172.,  nan,  nan,  nan,  nan,  nan],
       [ nan, 181., 178., 176.,  nan,  nan,  nan,  nan,  nan]])
Coordinates:
  * lat      (lat) float64 64B 8.0 12.0 16.0 20.0 24.0 28.0 32.0 36.0
  * lon      (lon) float64 72B 68.0 72.0 76.0 80.0 84.0 88.0 92.0 96.0 100.0

In [96]:
md_30["std_er_15"]

KeyError: 'std_er_15'